# Adım 4: Keşifsel Veri Analizi (EDA)
**Kişi 2 sorumluluğu** — `feature/spark-eda` branch

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, isnan, avg
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

GOLD_PATH = './delta_lake/gold'
PLOTS_DIR = './plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

def create_spark():
    return (
        SparkSession.builder
        .appName('ClimateEDA')
        .master('local[*]')
        .config('spark.sql.extensions',
                'io.delta.sql.DeltaSparkSessionExtension')
        .config('spark.sql.catalog.spark_catalog',
                'org.apache.spark.sql.delta.catalog.DeltaCatalog')
        .config('spark.jars.packages',
                'io.delta:delta-core_2.12:2.4.0')
        .config('spark.sql.execution.arrow.pyspark.enabled', 'false')
        .getOrCreate()
    )

spark = create_spark()
spark.sparkContext.setLogLevel('WARN')
df = spark.read.format('delta').load(GOLD_PATH)
print(f'Gold tablodan {df.count():,} kayit yuklendi.')

In [ ]:
pdf = df.select('avg_temp_c', 'min_temp_c', 'max_temp_c',
                'precipitation_mm', 'avg_wind_speed_kmh').describe().toPandas()
print('[EDA] Temel Istatistikler:')
print(pdf.to_string())

In [ ]:
cols = ['avg_temp_c', 'min_temp_c', 'max_temp_c', 'precipitation_mm',
        'snow_depth_mm', 'avg_wind_speed_kmh', 'avg_sea_level_pres_hpa',
        'sunshine_total_min']
missing = {c: df.filter(col(c).isNull() | isnan(col(c))).count() for c in cols}
total = df.count()
missing_pct = {k: round(v / total * 100, 2) for k, v in missing.items()}

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(missing_pct.keys(), missing_pct.values(),
              color=sns.color_palette('Reds_r', len(cols)))
ax.set_title('Sutun Bazli Eksik Deger Orani (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Eksik Deger (%)')
ax.set_xlabel('Sutunlar')
ax.bar_label(bars, fmt='%.1f%%', padding=3)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/01_missing_values.png')
plt.show()
print('[Gorsel 1] Eksik deger grafigi kaydedildi.')

## EDA — Kalan Görseller

In [ ]:
pdf = (df.groupBy('year').agg(avg('avg_temp_c').alias('ort_sicaklik'))
       .orderBy('year').toPandas())
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(pdf['year'], pdf['ort_sicaklik'], marker='o', linewidth=2, color='#e63946')
ax.fill_between(pdf['year'], pdf['ort_sicaklik'], alpha=0.15, color='#e63946')
ax.set_title('Yillik Ortalama Kuresel Sicaklik Trendi', fontsize=13, fontweight='bold')
ax.set_xlabel('Yil'); ax.set_ylabel('Ortalama Sicaklik (°C)')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/02_yearly_trend.png'); plt.show()
print('[Gorsel 2] Yillik sicaklik trendi kaydedildi.')

In [ ]:
ay_isimleri = ['Oca','Sub','Mar','Nis','May','Haz','Tem','Agu','Eyl','Eki','Kas','Ara']
pdf = (df.groupBy('month').agg(avg('avg_temp_c').alias('ort_sicaklik'))
       .orderBy('month').toPandas())
pdf['ay'] = pdf['month'].apply(lambda x: ay_isimleri[x - 1])
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=pdf, x='ay', y='ort_sicaklik', palette='coolwarm', ax=ax)
ax.set_title('Aylik Ortalama Sicaklik Dagilimi', fontsize=13, fontweight='bold')
ax.set_xlabel('Ay'); ax.set_ylabel('Ortalama Sicaklik (°C)')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/03_monthly_avg.png'); plt.show()
print('[Gorsel 3] Aylik sicaklik dagilimi kaydedildi.')

In [ ]:
pdf = (df.groupBy('city_name').agg(avg('avg_temp_c').alias('ort'))
       .orderBy(col('ort').desc()).limit(20).toPandas())
fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=pdf, y='city_name', x='ort', palette='YlOrRd', ax=ax)
ax.set_title('En Sicak 20 Sehir (Ortalama Sicaklik)', fontsize=13, fontweight='bold')
ax.set_xlabel('Ortalama Sicaklik (°C)'); ax.set_ylabel('Sehir')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/04_top20_cities.png'); plt.show()
print('[Gorsel 4] Top 20 sehir grafigi kaydedildi.')

In [ ]:
num_cols = ['avg_temp_c', 'min_temp_c', 'max_temp_c',
            'precipitation_mm', 'avg_wind_speed_kmh',
            'avg_sea_level_pres_hpa', 'sunshine_total_min']
pdf = df.select(num_cols).dropna().limit(50000).toPandas()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(pdf.corr(), annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Sayisal Degiskenler Arasi Korelasyon Matrisi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/05_correlation.png'); plt.show()
print('[Gorsel 5] Korelasyon matrisi kaydedildi.')

In [ ]:
pdf = df.select('avg_temp_c').dropna().limit(100000).toPandas()
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(pdf['avg_temp_c'], bins=80, color='#457b9d', edgecolor='white', alpha=0.85)
ax.set_title('Gunluk Ortalama Sicaklik Dagilimi', fontsize=13, fontweight='bold')
ax.set_xlabel('Ortalama Sicaklik (°C)'); ax.set_ylabel('Frekans')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/06_temp_histogram.png'); plt.show()
print('[Gorsel 6] Sicaklik histogram kaydedildi.')

In [ ]:
pdf = df.select('season', 'avg_temp_c').dropna().limit(50000).toPandas()
season_order = ['Winter', 'Spring', 'Summer', 'Autumn']
season_tr = {'Winter': 'Kis', 'Spring': 'Ilkbahar', 'Summer': 'Yaz', 'Autumn': 'Sonbahar'}
pdf['season_tr'] = pdf['season'].map(season_tr)
fig, ax = plt.subplots(figsize=(9, 6))
sns.boxplot(data=pdf, x='season_tr', y='avg_temp_c',
            order=[season_tr[s] for s in season_order], palette='Set2', ax=ax)
ax.set_title('Mevsimsel Sicaklik Dagilimi (Box Plot)', fontsize=13, fontweight='bold')
ax.set_xlabel('Mevsim'); ax.set_ylabel('Ortalama Sicaklik (°C)')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/07_seasonal_boxplot.png'); plt.show()
print('[Gorsel 7] Mevsimsel box plot kaydedildi.')

In [ ]:
pdf = df.select('avg_temp_c', 'precipitation_mm', 'season').dropna().limit(20000).toPandas()
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(pdf['avg_temp_c'], pdf['precipitation_mm'],
           c=pd.Categorical(pdf['season']).codes,
           cmap='tab10', alpha=0.4, s=8)
ax.set_title('Sicaklik — Yagis Iliskisi', fontsize=13, fontweight='bold')
ax.set_xlabel('Ortalama Sicaklik (°C)'); ax.set_ylabel('Yagis (mm)')
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/08_precip_vs_temp.png'); plt.show()
print('[Gorsel 8] Yagis-sicaklik scatter kaydedildi.')

print(f"\nTum gorseller '{PLOTS_DIR}' klasorune kaydedildi.")
spark.stop()